<a href="https://colab.research.google.com/github/Vania-cortina/Reconocimiento-de-Emociones/blob/main/Proyecto_An%C3%A1lisis_de_Sentimiento_CargarImagen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalar dependencias y configurar Kaggle

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar librerías necesarias
!pip install tensorflow keras opencv-python-headless numpy pandas scikit-learn matplotlib seaborn kaggle streamlit pyngrok streamlit-webrtc deepface

# Configurar Kaggle
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


/bin/bash: line 1: nvidia-smi: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.3/128.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.6 MB/s eta 0:00:00
   ━━━

Descargar y descomprimir datasets

In [ ]:
# Crear carpeta para datos
os.makedirs('/content/data/raw', exist_ok=True)

datasets = [
    "shawon10/ckplus",
    "msambare/fer2013",
    "ashwingupta3012/rafdb"
]

for d in datasets:
    print("Descargando:", d)
    !kaggle datasets download -d {d} -p /content/data/raw --unzip

print("Descargas completas.")


Descargando: shawon10/ckplus
Dataset URL: https://www.kaggle.com/datasets/shawon10/ckplus
License(s): CC0-1.0
  0% 0.00/3.63M [00:00<?, ?B/s]
100% 3.63M/3.63M [00:00<00:00, 1.07GB/s]
Descargando: msambare/fer2013
Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 1.00GB/s]
Descargando: ashwingupta3012/rafdb
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/ashwingupta3012/rafdb
Descargas completas.


Funciones de preprocesamiento y carga

In [ ]:
import cv2
import numpy as np
import os
from glob import glob
from tqdm import tqdm
import pandas as pd

TARGET_SIZE = (48,48)

def preprocess_img(img_path, target_size=TARGET_SIZE):
    img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_COLOR) if isinstance(img_path, str) else img_path
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, target_size)
    normalized = resized.astype('float32') / 255.0
    return normalized

def load_fer2013(csv_path='/content/fer2013.csv'):
    if not os.path.exists(csv_path):
        print('FER2013 CSV no encontrado en', csv_path)
        return np.array([]), np.array([])
    df = pd.read_csv(csv_path)
    X, y = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        pixels = np.fromstring(row['pixels'], sep=' ', dtype='float32')
        if pixels.size != 48*48:
            continue
        img = pixels.reshape(48,48)/255.0
        X.append(img)
        y.append(int(row['emotion']))
    X = np.array(X).reshape(-1,48,48,1)
    y = np.array(y)
    print('FER2013 cargado:', X.shape, y.shape)
    return X, y

# Cargar datos
X, y = load_fer2013('/content/fer2013.csv')

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)


100%|██████████| 35887/35887 [00:19<00:00, 1874.62it/s]


FER2013 cargado: (35887, 48, 48, 1) (35887,)
Train: (28709, 48, 48, 1) Val: (3589, 48, 48, 1) Test: (3589, 48, 48, 1)


Definir y entrenar el modelo

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

num_classes = 7

def build_emotion_model(input_shape=(48,48,1), num_classes=7):
    model = Sequential([
        Conv2D(32,(3,3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Conv2D(64,(3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Conv2D(128,(3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Flatten(),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_emotion_model()
model.summary()

# Data augmentation
batch_size = 64
epochs = 25
train_gen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1,
                               zoom_range=0.1, horizontal_flip=True).flow(X_train, y_train, batch_size=batch_size)
val_gen = ImageDataGenerator().flow(X_val, y_val, batch_size=batch_size)

# Callbacks
checkpoint = ModelCheckpoint('/content/facial_emotion_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
early = EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)

# Entrenamiento
history = model.fit(train_gen, epochs=epochs, validation_data=val_gen, callbacks=[checkpoint, early])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,743 (1.36 MB)

 Trainable params: 356,295 (1.36 MB)

...

449/449 ━━━━━━━━━━━━━━━━━━━━ 141s 313ms/step - accuracy: 0.5225 - loss: 1.2531 - val_accuracy: 0.5461 - val_loss: 1.1826
Epoch 11/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5290 - loss: 1.2297
Epoch 11: val_accuracy did not improve from 0.54611
449/449 ━━━━━━━━━━━━━━━━━━━━ 142s 316ms/step - accuracy: 0.5290 - loss: 1.2297 - val_accuracy: 0.5286 - val_loss: 1.2524
Epoch 12/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - accuracy: 0.5404 - loss: 1.2168
Epoch 12: val_accuracy did not improve from 0.54611
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 318ms/step - accuracy: 0.5404 - loss: 1.2168 - val_accuracy: 0.5258 - val_loss: 1.2139
Epoch 13/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - accuracy: 0.5473 - loss: 1.1934
Epoch 13: val_accuracy improved from 0.54611 to 0.55224, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 140s 312ms/step - accuracy: 0.5472 - loss: 1.1934 - val_accuracy: 0.5522 - val_loss: 1.1534
Epoch 14/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.5425 - loss: 1.1971
Epoch 14: val_accuracy improved from 0.55224 to 0.57843, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 141s 315ms/step - accuracy: 0.5425 - loss: 1.1971 - val_accuracy: 0.5784 - val_loss: 1.1153
Epoch 15/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - accuracy: 0.5494 - loss: 1.1858
Epoch 15: val_accuracy did not improve from 0.57843
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 318ms/step - accuracy: 0.5494 - loss: 1.1858 - val_accuracy: 0.5339 - val_loss: 1.2548
Epoch 16/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.5525 - loss: 1.1754
Epoch 16: val_accuracy did not improve from 0.57843
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 319ms/step - accuracy: 0.5525 - loss: 1.1754 - val_accuracy: 0.5684 - val_loss: 1.1201
Epoch 17/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - accuracy: 0.5518 - loss: 1.1691
Epoch 17: val_accuracy improved from 0.57843 to 0.58512, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 145s 324ms/step - accuracy: 0.5518 - loss: 1.1691 - val_accuracy: 0.5851 - val_loss: 1.0853
Epoch 18/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5474 - loss: 1.1721
Epoch 18: val_accuracy did not improve from 0.58512
449/449 ━━━━━━━━━━━━━━━━━━━━ 144s 321ms/step - accuracy: 0.5474 - loss: 1.1721 - val_accuracy: 0.5584 - val_loss: 1.1509
Epoch 19/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5637 - loss: 1.1562
Epoch 19: val_accuracy did not improve from 0.58512
449/449 ━━━━━━━━━━━━━━━━━━━━ 142s 317ms/step - accuracy: 0.5637 - loss: 1.1562 - val_accuracy: 0.4982 - val_loss: 1.3869
Epoch 20/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.5609 - loss: 1.1644
Epoch 20: val_accuracy did not improve from 0.58512
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 319ms/step - accuracy: 0.5609 - loss: 1.1644 - val_accuracy: 0.5559 - val_loss: 1.1856
Epoch 21/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5634 - loss: 1.1420
Epoc

449/449 ━━━━━━━━━━━━━━━━━━━━ 142s 316ms/step - accuracy: 0.5694 - loss: 1.1300 - val_accuracy: 0.5949 - val_loss: 1.0892
Epoch 23/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - accuracy: 0.5709 - loss: 1.1358
Epoch 23: val_accuracy did not improve from 0.59487
449/449 ━━━━━━━━━━━━━━━━━━━━ 144s 320ms/step - accuracy: 0.5709 - loss: 1.1358 - val_accuracy: 0.5837 - val_loss: 1.1038
Epoch 24/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5651 - loss: 1.1378
Epoch 24: val_accuracy did not improve from 0.59487
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 319ms/step - accuracy: 0.5651 - loss: 1.1378 - val_accuracy: 0.5876 - val_loss: 1.0951
Epoch 25/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.5750 - loss: 1.1259
Epoch 25: val_accuracy did not improve from 0.59487
449/449 ━━━━━━━━━━━━━━━━━━━━ 143s 318ms/step - accuracy: 0.5750 - loss: 1.1259 - val_accuracy: 0.5762 - val_loss: 1.1070


Crear app combinada Streamlit

In [ ]:
app_code = r'''
import streamlit as st
from tensorflow.keras.models import load_model
import numpy as np
from PIL import Image
import cv2
from deepface import DeepFace
from streamlit_webrtc import webrtc_streamer, VideoTransformerBase

# Cargar modelo entrenado
model = load_model('/content/facial_emotion_model.h5')
emotions = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']

st.title("Reconocimiento de Emociones — Demo 🎭")

# Subida de imagen
st.header("Analizar imagen subida")
uploaded_file = st.file_uploader("Sube una imagen", type=['png','jpg','jpeg'])
if uploaded_file:
    img = Image.open(uploaded_file).convert('RGB')
    st.image(img, caption='Imagen subida', use_column_width=True)
    st.write("Prediciendo con modelo entrenado...")

    img_arr = np.array(img)
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (48,48)).astype('float32')/255.0
    inp = resized.reshape(1,48,48,1)

    preds = model.predict(inp)[0]
    top_idx = preds.argmax()
    st.subheader(f"Emoción detectada: {emotions[top_idx]} (confianza: {preds[top_idx]:.2f})")
    st.bar_chart({'probabilidad': preds})

# Webcam en tiempo real con DeepFace
st.header("Analizar emociones en tiempo real (Webcam)")

class EmotionTransformer(VideoTransformerBase):
    def transform(self, frame):
        img = frame.to_ndarray(format="bgr24")
        try:
            result = DeepFace.analyze(img, actions=['emotion'], enforce_detection=False)
            dominant_emotion = result['dominant_emotion']
            cv2.putText(img, dominant_emotion, (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2, cv2.LINE_AA)
        except Exception:
            pass
        return img

webrtc_streamer(key="emotion", video_transformer_factory=EmotionTransformer)
'''

with open('/content/app.py','w') as f:
    f.write(app_code)

print('app.py creado en /content/app.py')


app.py creado en /content/app.py


Ejecutar Streamlit en Colab con ngrok

In [ ]:
!ngrok authtoken 35MJSCKYxRXSAiR2xYyJoa7eHrr_69PCWPSyN71ThsjrEaERg

from pyngrok import ngrok
import subprocess, os

# Cerrar túneles previos
ngrok.kill()

# Ejecutar Streamlit en background
cmd = "streamlit run /content/app.py --server.port 8501"
proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Abrir túnel ngrok
public_url = ngrok.connect(8501).public_url
print("Streamlit corriendo en:", public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Streamlit corriendo en: https://noneradicative-prefuneral-tobi.ngrok-free.dev
